In [ ]:
1. Задача про зарплаты футболистов. В файле football_players.csv представлен фрагмент выгрузки датасета о футболистах. Посчитайте среднюю и медианную зарплату "Wage" футболистов из разных клубов "Club". В скольких клубах средняя и медианная зарплаты совпадают?

In [11]:
import pandas as pd

football_players = pd.read_csv("files/lesson4/football_players.csv")
clubs = football_players.groupby('Club')["Wage"].agg(['mean', 'median'])
same_mean_median_clubs = clubs[clubs['mean'] == clubs['median']]
len(same_mean_median_clubs)

52

2. Задача нахождения сходства предложений в тексте. Необходимо написать программу, анализирующую сходство предложений в файле между собой

In [57]:
import re
from scipy.spatial.distance import cosine

file_path = 'files/lesson4/anaconda.txt'
sentence_separators = ['.', '!', '?']
ignore_symbols = [',', ':', '"', "'", '(', ')', '\n']
ignore_pattern = '|'.join(ignore_symbols)
split_pattern = '|'.join(re.escape(sep) for sep in sentence_separators)
ignore_words = {"a", "an", "the", "any", "by", "as", "for", "of", "is"}
replace_words = {"vs.": "versus", "e.g.": "example"}

with open(file_path, 'r') as file:
    # Приводим к нижнему регистру, чтобы считать слова одинаковыми независимо от заглавных букв (например, в начале предложения)
    text = file.read().lower()
    # Заменяем сокращения, чтобы не разбивать предложения, где точка - не знак конца предложения. И убираем не значимые слова
    for short, full in replace_words.items():
        text = text.replace(short, full)
# Игнорируем пунктуацию внутри предложения, чтобы получить чистые слова
clean_text = re.sub(f'[{ignore_pattern}]', '', text)
# Делим предложения по символам, которые заканчивают предложения - ['.', '!', '?']
clean_sentences = re.split(split_pattern, clean_text)

# Множество всех уникальных слов в тексте. Сортируем для детерминированности. Убираем "не значимые" слова
text_dictionary = sorted(set(clean_text.split()) - ignore_words)

# Так как косинусное расстояние считается для векторов одинаковой длины, приводим предложения в список повторений слов с учётом индексов по общему словарю
def sentence_similarity(text_dictionary: list[str], sentence_1: str, sentence_2: str):
    words_1 = sentence_1.split()
    words_2 = sentence_2.split()
    num_words_1 = {}
    num_words_2 = {}
    # Формируем соответствие {слово: количество повторений в предложении} для первого и второго предложения
    for word in words_1:
        if not word in num_words_1:
            num_words_1[word] = 1
        else:
            num_words_1[word] += 1
    for word in words_2:
        if not word in num_words_2:
            num_words_2[word] = 1
        else:
            num_words_2[word] += 1
    sentence_1_vector = [num_words_1.get(word, 0) for word in text_dictionary]
    sentence_2_vector = [num_words_2.get(word, 0) for word in text_dictionary]
    return cosine(sentence_1_vector, sentence_2_vector)

sentence_similarity(text_dictionary, clean_sentences[0], clean_sentences[1])

np.float64(0.75)